In [ ]:
# ==========================================
# 1. THE STABLE ENVIRONMENT INSTALL
# ==========================================
import os
# (Commented out so we don't disconnect the kernel again)
# os.system('pip install "torch>=2.4.0,<2.11.0" torchvision torchaudio')
# os.system('pip install unsloth unsloth_zoo')
# os.system('pip install "trl==0.22.0" peft accelerate bitsandbytes openenv-core datasets')

import re
import json
import torch
from tqdm import tqdm
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

from env import EnterpriseEmailEnv

# ==========================================
# 2. LOAD THE FAST 3B MODEL
# ==========================================
print("Loading Unsloth 3B Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", 
    max_seq_length = 512,
    dtype = None,
    load_in_4bit = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ==========================================
# 3. GENERATE ROLLOUTS (BULLETPROOF PARSER)
# ==========================================
print("Initializing OpenEnv...")
env = EnterpriseEmailEnv()

def format_observation_to_prompt(obs):
    """Refined prompt that uses high-authority instructions for 3B model stability."""
    current_email = obs.get('current_email', {})
    email_id = current_email.get('email_id', 'UNKNOWN_ID')
    sender = current_email.get('sender', 'Unknown')
    subject = current_email.get('subject', 'No Subject')
    body = current_email.get('body', '')[:1000] 
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Corporate Email Triage Agent. You must respond ONLY with a JSON object.

### AVAILABLE TOOLS:
1. {{"tool": "route_to_human", "arguments": {{"email_id": "{email_id}", "department": "IT/Support/HR"}}}}
   *RULE: Use this for VIPs, Angry customers, Server Outages, or Sensitive HR issues.*

2. {{"tool": "auto_reply", "arguments": {{"email_id": "{email_id}", "message": "REPLY_TEXT"}}}}
   *RULE: Use this for routine tasks like password resets or feature requests.*

3. {{"tool": "ask_for_clarification", "arguments": {{"email_id": "{email_id}"}}}}
   *RULE: Use this only if the email is confusing or missing data.*

Always use the correct Email ID: {email_id}
<|eot_id|><|start_header_id|>user<|end_header_id|>
Email ID: {email_id}
From: {sender}
Subject: {subject}
Body: {body}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    return prompt

def collect_rollouts(env, model, tokenizer, num_episodes=30):
    rollout_data = []
    FastLanguageModel.for_inference(model)

    for episode in tqdm(range(num_episodes), desc="Collecting rollouts"):
        obs = env.reset()
        episode_data = []

        while not env.current_step >= env.max_steps and len(env.processed_emails) < len(env.emails):
            prompt = format_observation_to_prompt(obs)
            inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=300, 
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

            start_idx = generated_text.find('{')
            end_idx = generated_text.rfind('}') + 1
            
            if start_idx != -1 and end_idx != 0:
                clean_json = generated_text[start_idx:end_idx]
                try:
                    action = json.loads(clean_json)
                    if "tool" not in action or "arguments" not in action:
                        action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
                except:
                    action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}
            else:
                action = {"tool": "ask_for_clarification", "arguments": {"email_id": obs['current_email']['email_id']}}

            next_obs, reward, done, info = env.step(action)

            episode_data.append({
                "prompt": prompt,
                "action": action,
                "reward": reward,
                "observation": next_obs
            })

            print(f"Step {env.current_step}: {action['tool']} -> Reward: {reward:+.2f} | Total: {env.total_reward:+.2f}")
            obs = next_obs
            if done: break

        rollout_data.extend(episode_data)
    
    FastLanguageModel.for_training(model)
    return rollout_data

print("Collecting rollouts (playing the game)...")
rollout_data = collect_rollouts(env, model, tokenizer, num_episodes=30)
print(f"Collected {len(rollout_data)} transitions")

# ==========================================
# 4. PREPARE BEHAVIORAL CLONING DATASET
# ==========================================
print("Filtering for successful behaviors...")
winning_rollouts = [item for item in rollout_data if item["reward"] > 0]
print(f"Found {len(winning_rollouts)} highly successful actions to train on!")

sft_dataset_dict = {
    "text": [item["prompt"] + json.dumps(item["action"]) + tokenizer.eos_token for item in winning_rollouts]
}
sft_dataset = Dataset.from_dict(sft_dataset_dict)

# ==========================================
# 5. INITIALIZE SFT TRAINER
# ==========================================
print("Configuring SFT Trainer...")
sft_config = TrainingArguments(
    output_dir="email-triage-lora-final",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=60, 
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    args=sft_config,
)

# ==========================================
# 6. EXECUTE TRAINING & SAVE DELIVERABLES
# ==========================================
print("Starting Behavioral Cloning Training...")
sft_trainer.train()

print("Training completed! Saving LoRA adapters...")
model.save_pretrained("email-triage-lora-final")
tokenizer.save_pretrained("email-triage-lora-final")

print("SUCCESS: Check your file browser for the 'email-triage-lora-final' folder!")

Loading Unsloth 3B Model...
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.301 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Initializing OpenEnv...
✅ Successfully loaded 100 emails from dataset.json


✅ Successfully loaded 100 emails from dataset.json


...

Step 5: ask_for_clarification -> Reward: -0.37 | Total: +0.04


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 6: route_to_human -> Reward: +0.83 | Total: +0.87


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 7: route_to_human -> Reward: +0.83 | Total: +1.70
